In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_106_IGI_Airport_(T3)_Delhi_IMD_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,159.71,271.39,29.13,8.42,37.55,NaN,NaN,1.58,23.16,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2024-01-02,169.18,260.73,27.01,15.83,42.62,NaN,NaN,1.11,19.94,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2024-01-03,176.04,282.83,30.61,23.08,53.58,NaN,NaN,0.77,11.05,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2024-01-04,184.63,294.51,33.27,13.87,47.10,NaN,NaN,0.62,7.59,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2024-01-05,133.00,209.96,30.82,19.66,50.43,NaN,NaN,1.06,11.22,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,123.37,176.29,62.50,51.02,77.95,NaN,NaN,0.76,14.56,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
362,2024-12-28,70.32,106.72,50.05,36.54,60.13,NaN,NaN,0.37,11.20,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
363,2024-12-29,68.62,101.67,18.05,23.00,26.91,NaN,NaN,0.13,21.87,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
364,2024-12-30,73.60,117.96,23.08,25.75,32.46,NaN,NaN,0.09,23.63,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 10)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['NH3 (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp        0
PM2.5 (µg/m³)    0
PM10 (µg/m³)     0
NO (µg/m³)       0
NO2 (µg/m³)      0
NOx (ppb)        0
CO (mg/m³)       0
Ozone (µg/m³)    0
TOT-RF (mm)      0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 9)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         159.71        271.39       29.13         8.42   
1  2024-01-02         169.18        260.73       27.01        15.83   
2  2024-01-03         176.04        282.83       30.61        23.08   
3  2024-01-04         184.63        294.51       33.27        13.87   
4  2024-01-05         133.00        209.96       30.82        19.66   

   NOx (ppb)  CO (mg/m³)  Ozone (µg/m³)  TOT-RF (mm)  
0      37.55        1.58          23.16          0.0  
1      42.62        1.11          19.94          0.0  
2      53.58        0.77          11.05          0.0  
3      47.10        0.62           7.59          0.0  
4      50.43        1.06          11.22          0.0  


In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),CO (mg/m³),Ozone (µg/m³),TOT-RF (mm)
0,2024-01-01,1.406561,0.890628,-0.244478,-1.521722,-0.820489,1.414325,0.479923,0.0
1,2024-01-02,1.576618,0.780721,-0.332221,-1.071297,-0.680189,0.419794,0.078108,0.0
2,2024-01-03,1.699806,1.008577,-0.183223,-0.630599,-0.376900,-0.299655,-1.031252,0.0
3,2024-01-04,1.854060,1.129001,-0.073131,-1.190438,-0.556217,-0.617058,-1.463016,0.0
4,2024-01-05,0.926919,0.257269,-0.174532,-0.838487,-0.464068,0.313993,-1.010038,0.0
...,...,...,...,...,...,...,...,...,...
361,2024-12-27,0.753989,-0.089877,1.136648,1.067763,0.297477,-0.320815,-0.593248,0.0
362,2024-12-28,-0.198652,-0.807161,0.621364,0.187581,-0.195645,-1.146064,-1.012534,0.0
363,2024-12-29,-0.229180,-0.859228,-0.703059,-0.635461,-1.114923,-1.653910,0.318948,0.0
364,2024-12-30,-0.139752,-0.691274,-0.494877,-0.468300,-0.961341,-1.738551,0.538573,0.0


In [10]:
df.to_excel('IGIAirport2024.xlsx', index=False)